## Imports

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from pyhive import presto
from datetime import datetime, date, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
#     processing-2.processing.data.production.internal
#     presto-gateway.serving.data.production.internal
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
start_date='2024-04-23'
end_date='2024-05-08'

In [5]:
city_name=['Bangalore']
service_level=['Auto','CabEconomy']

## Fetching data

In [22]:
def route_adherence_func(start_date, end_date, city_name, service_name):

    route_adherence_query = f"""    
    
        WITH service_mapping AS (

            SELECT 
                service_detail_id,
                service_level as service_name,
                service_category,
                service_id,
                city_display_name as city,
                city_id
            FROM 
                datasets.service_mapping
            WHERE 
                city_display_name = '{city_name}'
                and service_level = '{service_name}'
            ),

            fe_tbl as (
            
            SELECT
                fare_estimate_id fid,
                api_context,final_amount as final_amount_fare,
                date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS order_date,
                cast(final_amount AS double) AS fe_amount,service_detail_id as service_id_fare,
                epoch fe,ride_distance as d1, estimated_ride_time as t1
            FROM
                pricing.fare_estimates_enriched 
            WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                AND city IN (SELECT city FROM service_mapping)
                AND service_name IN (SELECT service_name FROM service_mapping)
            ),
        
        
        
        orderlogs as (
            
            SELECT 
                *,
                u.id_array as fare_estimate_id
            FROM 
                ( 
                SELECT 
                    order_id, (amount-prev_due) as order_amount,
                    customer_id,captain_id,
                    date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
                    epoch,estimate_id,estimate_ids,ride_time as t3,
                    fare_recalculated_reason, 
                    all_paths,
                    spd_fraud_flag,order_status,
                    service_obj_service_name as service_name,
                    fare_recalculated_diff_amount AS fe_re_diff_amount,service_detail_id,
                    
                    json_array_length(all_paths) AS generated_route,
                    cast(json_extract_scalar(all_paths, '$[0].distance') as double) /1000 AS route1_distance,
                    cast(json_extract_scalar(all_paths, '$[0].time') as double)/60 AS route1_time,
                    cast(json_extract_scalar(all_paths, '$[1].distance') as double) /1000 AS route2_distance,
                    cast(json_extract_scalar(all_paths, '$[1].time') as double) /60 AS route2_time,
                    cast(json_extract_scalar(all_paths, '$[2].distance') as double) /1000 AS route3_distance,
                    cast(json_extract_scalar(all_paths, '$[2].time') as double) /60 AS route3_time,
            
                    cast(json_parse(estimate_ids) as array<varchar>) AS id_array
                FROM 
                    orders.order_logs_snapshot
                WHERE 
                    yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                    AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                    AND city_name IN (SELECT city FROM service_mapping)
                    AND service_obj_service_name IN (SELECT service_name FROM service_mapping)
                ) as t
                CROSS JOIN UNNEST(t.id_array) AS u(id_array)
            ),
            
            immutable AS (
            
            SELECT 
                DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
                order_id,
                cast (event_time as double) event_time
            FROM 
                orders.order_logs_immutable 
            WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                AND event_type = 'started'
                AND city_name IN (SELECT city FROM service_mapping)
                AND service_obj_service_name IN (SELECT service_name FROM service_mapping)
            ),
        
        
            nav_metrics AS (
            
            SELECT
                *
            FROM
                (
                SELECT 
                    event_props_order_id,
                    case when device_info_app_version>='5.3.22' then 'Above 5.3.22' else 'Below 5.3.22' end as app_version_type,
                    case when device_info_app_version>='5.3.24' then 'Above 5.3.24' else 'Below 5.3.24' end as app_optimal_version_type,
                    cast  (event_props_epoch as double) event_props_epoch,
                    DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
                    try(cast(event_props_distance_in_meters as double) )/ 1000 d2,
                    row_number() over(partition by yyyymmdd, event_props_order_id order by event_props_epoch) rn
                FROM
                    clevertap.captain_inappnavmetrics_immutable
                WHERE 
                    yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                AND event_props_order_status = 'started'
                AND event_props_city IN  (SELECT city FROM service_mapping)
                )        
            WHERE 
                rn = 1
            ), 

            distance2 AS (
            
            SELECT 
                orderdate,
                order_id,
                app_version_type,
                app_optimal_version_type,
                d2
            FROM 
                ( 
                SELECT 
                    imu.orderdate,
                    imu.order_id,
                    app_version_type,
                     app_optimal_version_type,
                    date_diff('minute', from_unixtime(event_time/1000), from_unixtime(event_props_epoch/1000)) AS minDiff,
                    "d2"
                FROM 
                    immutable imu 
                JOIN 
                    nav_metrics nav 
                    ON imu.order_id = nav.event_props_order_id
                )
                --and event_props_epoch >= event_time )
                
            WHERE abs(minDiff) <= 1 -- Relaxing threshold
            
            ),
        
            done as (

            SELECT 
                DISTINCT order_id,
                d1,t1,t3,
                epoch,fare_recalculated_reason,
                service_name,rn,
                api_context, (
                fe_amount-order_amount) as amt_diff, 
                all_paths,fid,
                spd_fraud_flag,
                order_status,
                orderdate,
                generated_route
            FROM 
                (
                SELECT 
                    *,
                    row_number () over(partition by order_id order by fe) rn 
                FROM 
                    (
                    SELECT 
                        *
                    FROM 
                        orderlogs 
                    JOIN 
                        fe_tbl
                        ON fare_estimate_id = fid 
                        AND service_detail_id = service_id_fare
                    )
                )
            WHERE rn =1
            ),
            
            
            dthree as (
            
            SELECT 
                DISTINCT order_id, 
                CASE
                    WHEN selectedroute = 'hFRoute' THEN hfdistance
                    WHEN selectedroute = 'actualRoute' THEN directionsresponsedistance
                    WHEN selectedroute = 'estimatedRoute' THEN final_estimated_directions_response_distance
                    ELSE NULL -- Handle other cases if needed
                END AS d3, 
                selectedroute
            FROM
                (
                SELECT 
                    * 
                FROM 
                    (
                    SELECT  
                        *, 
                        row_number () over(partition by order_id order by updatedAt) rn 
                    FROM 
                        (
                         SELECT 
                             yyyymmdd,orderid as order_id, 
                             cast(hfdistance as double)*1000 as hfdistance,
                             cast(pingdensity as double)*100 as pingdensity,
                             cast(directionsresponsedistance as double) as directionsresponsedistance,
                             cast(final_estimated_directions_response_distance as double) as final_estimated_directions_response_distance,
                             selectedroute,updatedAt
                         FROM 
                             raw.captain_distance_response
                         WHERE 
                             yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                             AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                        )
                    )
                WHERE 
                    rn = 1
                )
            ),
        
    
            hap_category_tbl as  (

            select  
                customer,
                order_id,
                rider,
                yyyymmdd,
                service,
                pickup_selection_mode,
                case 
                    when cust_set_pickup_to_ride_started_dist <= 30 then 'a. 0-30' 
                    when cust_set_pickup_to_ride_started_dist > 30 and cust_set_pickup_to_ride_started_dist<=50 then 'b. 30-50' 
                    when cust_set_pickup_to_ride_started_dist > 50 then 'c. >50'
                    else 'd. Null'
                end as hap_category
            from 
                reports_internal.customer_experience_location_metrics
            WHERE
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                AND order_status = 'dropped'
                AND city IN (SELECT city FROM service_mapping)
                AND service IN (SELECT service_name FROM service_mapping)
                
            ),

            merged_tbl as (

            SELECT 
                done.order_id,
                done.orderdate,
                d2,d3,d1,
                app_version_type,
                app_optimal_version_type,
                fare_recalculated_reason,spd_fraud_flag,
                order_status,service_name,
                CASE
                WHEN d2 IS NOT NULL then d1-d2 END AS fe_vs_nav,
                d1-(d3*1.00/1000.00) as Est_Nav,
                CASE WHEN d2 IS NOT NULL then d2-(d3*1.00/1000.00) END AS nav_actual,
                pickup_selection_mode,
                hap_category,
                generated_route
            FROM 
                done
            LEFT JOIN 
                dthree
                ON done.order_id = dthree.order_id
            LEFT JOIN distance2
                ON done.order_id = distance2.order_id
            LEFT JOIN hap_category_tbl
                ON done.order_id = hap_category_tbl.order_id
            )
            

            SELECT 
                service_name,
                orderdate,
                CASE 
                    WHEN orderdate >= ('2024-04-23') and (orderdate<='2024-04-28') then 'pre' 
                    WHEN orderdate>=('2024-05-03') THEN 'post' end as new_date_type,
                app_version_type,
                pickup_selection_mode,

                count(distinct order_id) as net_orders,

                count(distinct case when fare_recalculated_reason='routeDiff' then order_id end) as route_diff_orders,
                -- try(count(distinct case when fare_recalculated_reason='routeDiff' then order_id end)*100.00/count(distinct order_id)) as route_diff_orders_percent,

                count(distinct case when fe_vs_nav_bucket='a. Within_500_m' then order_id end) as fe_nav_adherence,
                count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and hap_category='a. 0-30' then order_id end) adherence_and_good_hap_num,
                count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and hap_category='b. 30-50' then order_id end) adherence_and_30_50_hap_num,
                count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and hap_category='c. >50' then order_id end) adherence_and_bad_hap_num,


                -- try(count(distinct case when fe_vs_nav_bucket='a. Within_500_m' then order_id end)*100.00/count(distinct order_id)) as fe_nav_adherence_percent,
                -- try(count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and hap_category='a. 0-30' then order_id end)*100.00/count(distinct case when fe_vs_nav_bucket='a. Within_500_m' then order_id end)) as adherence_and_good_hap,
                -- try(count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and hap_category='b. 30-50' then order_id end)*100.00/count(distinct case when fe_vs_nav_bucket='a. Within_500_m' then order_id end)) as adherence_and_30_50_hap,
                -- try(count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and hap_category='c. >50' then order_id end)*100.00/count(distinct case when fe_vs_nav_bucket='a. Within_500_m' then order_id end)) as adherence_and_bad_hap,


                count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') then order_id end) as fe_nav_non_adherence,
                count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and hap_category='a. 0-30' then order_id end) non_adherence_and_good_hap_num,
                count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and hap_category='b. 30-50' then order_id end) non_adherence_and_30_50_hap,
                count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and hap_category='c. >50' then order_id end) non_adherence_and_bad_hap,


                -- try(count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') then order_id end)*100.00/count(distinct order_id)) as fe_nav_non_adherence_percent,
                -- try(count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and hap_category='a. 0-30' then order_id end)*100.00/count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') then order_id end)) as non_adherence_and_good_hap,
                -- try(count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and hap_category='b. 30-50' then order_id end)*100.00/count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') then order_id end)) as non_adherence_and_30_50_hap,
                -- try(count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and hap_category='c. >50' then order_id end)*100.00/count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') then order_id end)) as non_adherence_and_bad_hap
                
                
                count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and generated_route = 1 then order_id end) as adherence_one_route_generate,
                count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and generated_route = 2 then order_id end) as adherence_two_route_generate,
                count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and generated_route = 3 then order_id end) as adherence_three_route_generate,
                count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and generated_route = 1 then order_id end) as non_adherence_one_route_generate,
                count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and generated_route = 2 then order_id end) as non_adherence_two_route_generate,
                count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and generated_route = 3 then order_id end) as non_adherence_three_route_generate,
                
                count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and generated_route = 1 and fare_recalculated_reason='routeDiff' then order_id end) as routeDiff_adherence_one_route_generate,
        count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and generated_route = 2 and fare_recalculated_reason='routeDiff' then order_id end) as routeDiff_adherence_two_route_generate,
        count(distinct case when fe_vs_nav_bucket='a. Within_500_m' and generated_route = 3 and fare_recalculated_reason='routeDiff' then order_id end) as routeDiff_adherence_three_route_generate,
        count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and generated_route = 1 and fare_recalculated_reason='routeDiff' then order_id end) as routeDiff_non_adherence_one_route_generate,
        count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and generated_route = 2 and fare_recalculated_reason='routeDiff' then order_id end) as routeDiff_non_adherence_two_route_generate,
        count(distinct case when fe_vs_nav_bucket in ('b. Less_than_500m', 'c. Greater_than_500m') and generated_route = 3 and fare_recalculated_reason='routeDiff' then order_id end) as routeDiff_non_adherence_three_route_generate



            FROM
                (
                SELECT 
                    *,
                    CASE 
                        WHEN d2 is not NULL and fe_vs_nav>=-0.5 and fe_vs_nav<=0.5 then 'a. Within_500_m'
                        WHEN d2 is not NULL and fe_vs_nav<-0.5 then 'b. Less_than_500m'
                        WHEN d2 is not NULL and fe_vs_nav>0.5 then 'c. Greater_than_500m' end as fe_vs_nav_bucket

                FROM
                    merged_tbl
                -- WHERE 
                --     app_version_type = 'Above 5.3.22' 
                GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
                )

            GROUP BY 1,2,3,4,5
            ORDER BY 1,2,6 desc
            
            
    """
    df_route_adherence = pd.read_sql(route_adherence_query,connection)
    
    return df_route_adherence

In [23]:
total_df_d1_d2 = pd.DataFrame()


for city in city_name:
    print(city)
    
    for service_name in service_level:
        print(service_name)
        
        df_temp = route_adherence_func(start_date,end_date, city, service_name)
        
#         df_temp['distance_type']='second_longest'
#         df_temp['fe_nav_bucket_within_500m_percent']=((df_temp['fe_nav_bucket_within_500m']*100.00)/df_temp['net_orders']).round(2)
#         df_temp['nav_actual_bucket_within_500m_percent']=((df_temp['nav_actual_bucket_within_500m']*100.00)/df_temp['net_orders']).round(2)
#         df_temp['route_diff_percent']=((df_temp['route_diff_orders']*100.00)/df_temp['net_orders']).round(2)
        
        total_df_d1_d2=pd.concat([total_df_d1_d2,df_temp])

Bangalore
Auto
CabEconomy


## Analysis 

In [29]:
total_df_d1_d2['service_name'].unique()

array(['Auto', 'CabEconomy'], dtype=object)

In [30]:
total_df_d1_d2.shape

(1271, 27)

In [31]:
total_df_d1_d2.head(5)

,service_name,orderdate,new_date_type,app_version_type,pickup_selection_mode,net_orders,route_diff_orders,fe_nav_adherence,adherence_and_good_hap_num,adherence_and_30_50_hap_num,adherence_and_bad_hap_num,fe_nav_non_adherence,non_adherence_and_good_hap_num,non_adherence_and_30_50_hap,non_adherence_and_bad_hap,adherence_one_route_generate,adherence_two_route_generate,adherence_three_route_generate,non_adherence_one_route_generate,non_adherence_two_route_generate,non_adherence_three_route_generate,routeDiff_adherence_one_route_generate,routeDiff_adherence_two_route_generate,routeDiff_adherence_three_route_generate,routeDiff_non_adherence_one_route_generate,routeDiff_non_adherence_two_route_generate,routeDiff_non_adherence_three_route_generate
0,Auto,2024-04-23,pre,None,None,196542,30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Auto,2024-04-23,pre,Below 5.3.22,gps,53511,13104,43610,33476,4754,5380,9901,6649,1298,1954,15472,10636,17502,3713,2562,3626,2562,2180,4303,1401,1126,1532
2,Auto,2024-04-23,pre,Above 5.3.22,gps,22616,5488,19555,14825,2209,2521,3061,2095,364,602,7161,4835,7559,1129,786,1146,1143,990,1837,540,413,565
3,Auto,2024-04-23,pre,Below 5.3.22,google_search,11077,2672,8881,5975,1160,1746,2196,1153,350,693,3559,2041,3281,902,529,765,582,383,798,341,248,320
4,Auto,2024-04-23,pre,Below 5.3.22,recent_history,8413,1917,6870,4756,884,1230,1543,922,193,428,2761,1578,2531,679,379,485,429,322,556,257,152,201


In [32]:
total_df_d1_d2

,service_name,orderdate,new_date_type,app_version_type,pickup_selection_mode,net_orders,route_diff_orders,fe_nav_adherence,adherence_and_good_hap_num,adherence_and_30_50_hap_num,adherence_and_bad_hap_num,fe_nav_non_adherence,non_adherence_and_good_hap_num,non_adherence_and_30_50_hap,non_adherence_and_bad_hap,adherence_one_route_generate,adherence_two_route_generate,adherence_three_route_generate,non_adherence_one_route_generate,non_adherence_two_route_generate,non_adherence_three_route_generate,routeDiff_adherence_one_route_generate,routeDiff_adherence_two_route_generate,routeDiff_adherence_three_route_generate,routeDiff_non_adherence_one_route_generate,routeDiff_non_adherence_two_route_generate,routeDiff_non_adherence_three_route_generate
0,Auto,2024-04-23,pre,None,None,196542,30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Auto,2024-04-23,pre,Below 5.3.22,gps,53511,13104,43610,33476,4754,5380,9901,6649,1298,1954,15472,10636,17502,3713,2562,3626,2562,2180,4303,1401,1126,1532
2,Auto,2024-04-23,pre,Above 5.3.22,gps,22616,5488,19555,14825,2209,2521,3061,2095,364,602,7161,4835,7559,1129,786,1146,1143,990,1837,540,413,565
3,Auto,2024-04-23,pre,Below 5.3.22,google_search,11077,2672,8881,5975,1160,1746,2196,1153,350,693,3559,2041,3281,902,529,765,582,383,798,341,248,320
4,Auto,2024-04-23,pre,Below 5.3.22,recent_history,8413,1917,6870,4756,884,1230,1543,922,193,428,2761,1578,2531,679,379,485,429,322,556,257,152,201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
612,CabEconomy,2024-05-08,post,Above 5.3.22,walk_and_save,8,5,4,2,0,2,4,2,0,2,1,2,1,2,1,1,0,1,0,2,1,1
613,CabEconomy,2024-05-08,post,Below 5.3.22,pinMovement,4,1,1,1,0,0,3,2,0,1,0,1,0,2,1,0,0,0,0,1,0,0
614,CabEconomy,2024-05-08,post,Below 5.3.22,favourites,4,2,3,2,0,1,1,0,1,0,0,2,1,0,0,1,0,1,1,0,0,0
615,CabEconomy,2024-05-08,post,Above 5.3.22,static_recos,3,3,2,0,0,2,1,0,0,1,0,0,2,0,0,1,0,0,2,0,0,1


In [33]:
total_df_d1_d2.columns

Index(['service_name', 'orderdate', 'new_date_type', 'app_version_type',
       'pickup_selection_mode', 'net_orders', 'route_diff_orders',
       'fe_nav_adherence', 'adherence_and_good_hap_num',
       'adherence_and_30_50_hap_num', 'adherence_and_bad_hap_num',
       'fe_nav_non_adherence', 'non_adherence_and_good_hap_num',
       'non_adherence_and_30_50_hap', 'non_adherence_and_bad_hap',
       'adherence_one_route_generate', 'adherence_two_route_generate',
       'adherence_three_route_generate', 'non_adherence_one_route_generate',
       'non_adherence_two_route_generate',
       'non_adherence_three_route_generate',
       'routeDiff_adherence_one_route_generate',
       'routeDiff_adherence_two_route_generate',
       'routeDiff_adherence_three_route_generate',
       'routeDiff_non_adherence_one_route_generate',
       'routeDiff_non_adherence_two_route_generate',
       'routeDiff_non_adherence_three_route_generate'],
      dtype='object')

In [36]:
total_df_d1_d2.to_clipboard(index=False)

In [35]:
total_df_d1_d2.to_csv('d1_new_v2_pickup_selection_mode_generated_route.csv', index=False)

In [18]:
df_k1 = total_df_d1_d2\
        .groupby(['service_name', 'new_date_type', 'app_version_type'])\
        .agg({'gross_orders': 'sum',
              'net_orders': 'sum',
              'route_diff_orders': 'sum',
              'fe_nav_bucket_within_500m': 'sum',
              'nav_actual_bucket_within_500m': 'sum',
              'Est_Nav_bucket_within_500m': 'sum',
              'Est_Nav_bucket_less_500m': 'sum',
              'Est_Nav_bucket_grt_500m': 'sum'
             })\
        .reset_index()
df_k1

,service_name,new_date_type,app_version_type,gross_orders,net_orders,route_diff_orders,fe_nav_bucket_within_500m,nav_actual_bucket_within_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m
0,Auto,post,Above 5.3.22,565969,564004,116298,520107,435714,430572,41771,90902
1,Auto,post,Below 5.3.22,24532,24429,5441,20496,18577,18202,1679,4526
2,Auto,pre,Above 5.3.22,383023,381814,81368,337259,290340,290289,35424,56065
3,Auto,pre,Below 5.3.22,419096,417787,90461,367926,315530,315218,38247,64297
4,CabEconomy,post,Above 5.3.22,94308,94198,27028,83976,68059,65191,13521,14850
5,CabEconomy,post,Below 5.3.22,2604,2599,915,1966,1760,1621,389,569
6,CabEconomy,pre,Above 5.3.22,61701,61625,19489,51901,42741,40624,11436,9559
7,CabEconomy,pre,Below 5.3.22,62993,62924,20707,52872,43106,40778,11874,10270


In [19]:
df_k1['Est_Nav_bucket_less_500m_percent']=(df_k1['Est_Nav_bucket_less_500m']*100.00)/(df_k1['net_orders'])
df_k1['Est_Nav_bucket_grt_500m_percent']=(df_k1['Est_Nav_bucket_grt_500m']*100.00)/(df_k1['net_orders'])
df_k1['fe_nav_bucket_within_500m_percent']=(df_k1['fe_nav_bucket_within_500m']*100.00)/df_k1['net_orders']
df_k1['nav_actual_bucket_within_500m_percent']=(df_k1['nav_actual_bucket_within_500m']*100.00)/df_k1['net_orders']
df_k1['route_diff_percent']=(df_k1['route_diff_orders']*100.00)/df_k1['net_orders']

In [20]:
df_k1

,service_name,new_date_type,app_version_type,gross_orders,net_orders,route_diff_orders,fe_nav_bucket_within_500m,nav_actual_bucket_within_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m,Est_Nav_bucket_less_500m_percent,Est_Nav_bucket_grt_500m_percent,fe_nav_bucket_within_500m_percent,nav_actual_bucket_within_500m_percent,route_diff_percent
0,Auto,post,Above 5.3.22,565969,564004,116298,520107,435714,430572,41771,90902,7.406153,16.117262,92.216899,77.253707,20.620067
1,Auto,post,Below 5.3.22,24532,24429,5441,20496,18577,18202,1679,4526,6.872979,18.527160,83.900282,76.044865,22.272709
2,Auto,pre,Above 5.3.22,383023,381814,81368,337259,290340,290289,35424,56065,9.277816,14.683851,88.330706,76.042261,21.310900
3,Auto,pre,Below 5.3.22,419096,417787,90461,367926,315530,315218,38247,64297,9.154665,15.389900,88.065450,75.524131,21.652421
4,CabEconomy,post,Above 5.3.22,94308,94198,27028,83976,68059,65191,13521,14850,14.353808,15.764666,89.148390,72.251003,28.692754
5,CabEconomy,post,Below 5.3.22,2604,2599,915,1966,1760,1621,389,569,14.967295,21.893036,75.644479,67.718353,35.205848
6,CabEconomy,pre,Above 5.3.22,61701,61625,19489,51901,42741,40624,11436,9559,18.557404,15.511562,84.220690,69.356592,31.625152
7,CabEconomy,pre,Below 5.3.22,62993,62924,20707,52872,43106,40778,11874,10270,18.870383,16.321276,84.025173,68.504863,32.907952


## Numbers

In [21]:
total_df_d1_d2.to_csv('d1_new_v2.csv', index=False)

In [22]:
total_df_d1_d2.shape

(84, 23)